# Protocol B publication analysis
Attach the release code dataset and PB_Final_Verified_Inputs.zip. Run on CPU. The output uses the 215 known test groups.


In [ ]:
# Install small analysis dependencies. Keep Kaggle's torch and torchvision.
import importlib.metadata as metadata
import subprocess
import sys
required = {'numpy': '2.3.5', 'pandas': '3.0.1', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0', 'python-docx': '1.2.0', 'reportlab': '4.4.9', 'Pillow': '13.0.0'}
changes = []
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        changes.append(f"{package}=={version}")
if changes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *changes], check=True)
print('Setup complete. The analysis starts in a fresh process.')


In [ ]:
from pathlib import Path
import json
import os
CODE = Path('/kaggle/working/pb_final_code')
from pathlib import Path
import subprocess
import sys

code_tools = sorted(Path('/kaggle/input').rglob('materialize_workflow.py'))
code_tools = [p for p in code_tools if (p.parents[1] / 'reproducibility/code_sources.csv').is_file()]
if len(code_tools) != 1:
    raise FileNotFoundError('Attach one extracted release code dataset. See docs/reproduce.md.')
subprocess.run([sys.executable, str(code_tools[0]), 'protocol_b_analysis', str(CODE)], check=True)
EXPECTED_FINAL_SHA = 'aa5c843f9c3eddb09b74c39c548496f33a13af56f7aab395f98e125e730431f8'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


In [ ]:
# Run in a fresh Python process; save a complete log as a notebook output.
command = [sys.executable, str(CODE / 'final_analysis.py'),
           '--input-root', '/kaggle/input', '--work', '/kaggle/working',
           '--expected-final-sha', EXPECTED_FINAL_SHA, '--bootstrap', '2000']
log_path = Path('/kaggle/working/final_analysis_run.log')
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'Analysis stopped with exit code {return_code}. Inspect the final error above and {log_path.name}.')


In [ ]:
from IPython.display import display, FileLink
report = json.loads(Path('/kaggle/working/PB_Final_Publication/report.json').read_text())
assert report['completed_jobs'] == 20 and report['test_groups'] == 215 and report['calibration_groups_used'] == 138
print('FINAL ANALYSIS PASS')
print('Ensemble:', report['ensemble'])
display(FileLink('PB_Final_Publication_Outputs.zip'))
display(FileLink('PB_Final_Publication/tables/PB_Corrected_Publication_Tables.docx'))
display(FileLink('PB_Final_Publication/tables/PB_Corrected_Publication_Tables.pdf'))
